In [2]:
import boto3
import awswrangler as wr
import pandas as pd
import numpy as np

from pathlib import Path


pd.options.display.float_format = "{:,.2f}".format

# Confirm which IAM identity this notebook is running as
sts = boto3.client("sts", region_name="us-east-2")
print(sts.get_caller_identity()["Arn"])

arn:aws:iam::541974874359:user/jonno


In [3]:
# ---- Constants ----
REGION     = "us-east-2"
BUCKET     = "jonno-lucas-steve-bucket"
PROJECT    = "usd-aai540-group1"
SILVER_DB  = "aai540_silver"
GOLD_DB    = "aai540_gold"
ATHENA_OUT = f"s3://{BUCKET}/{PROJECT}/athena-results/"

# --- where the data lives / where the artifact goes ---
GOLD_S3 = f"s3://{BUCKET}/{PROJECT}/gold/model_training_matrix_foodsvc/"
MODEL_DIR = "model_artifacts"            # local scratch dir for the artifact

# --- modeling choices (the decisions we locked in) ---
TARGET = "food_svc_taxable_sales_usd"
DROP_COLS = [
    "food_svc_taxable_sales_usd",        # target
    "total_all_outlets_usd",             # LEAKAGE: target is a component of this
    "county_fips", "state_fips", "county_name", "period_id",  # identifiers
    "year",                              # temporal split key, not a feature
    # "total_expected_attendance",		 # misguided artifact from data collection
]
TRAIN_YEARS = [2015, 2016, 2017, 2018, 2019, 2020, 2021]
VAL_YEARS = [2022]
TEST_YEARS = [2023]
LOG_TARGET = True                        # fit on log1p(y), report in dollars
SEED = 42

# awswrangler picks up region from the boto3 default session
boto3.setup_default_session(region_name=REGION)

In [4]:
"""Pull down raw data"""
df = wr.s3.read_parquet(GOLD_S3, dataset=True)

print(f"shape: {df.shape}")
df.head()

shape: (2088, 25)


,county_fips,state_fips,county_name,latitude,longitude,land_area_sqmi,quarter,period_id,food_svc_taxable_sales_usd,total_all_outlets_usd,...,n_festivals,total_festival_attendance,total_wages_usd,avg_employment,establishment_count,population,median_household_income,median_age,bachelor_or_higher_pct,year
0,06007,06,Butte County,39.67,-121.60,"1,677.13",1,2015Q1,73088458,725610670,...,0,0,718585172,76218,7770,209470,68574,36.30,31.72,2015
1,06015,06,Del Norte County,41.74,-123.96,"1,229.68",1,2015Q1,5599963,54343322,...,0,0,68561665,7714,782,27293,66780,40.80,20.97,2015
2,06065,06,Riverside County,33.75,-115.99,"7,303.02",1,2015Q1,881417946,7960482590,...,0,0,7223106190,644370,54234,2449909,89672,36.70,25.08,2015
3,06069,06,San Benito County,36.61,-121.07,"1,390.47",1,2015Q1,13714308,130332097,...,0,0,158197461,14869,1492,66056,108289,35.90,22.39,2015
4,06085,06,Santa Clara County,37.23,-121.70,"1,304.06",1,2015Q1,1042756339,9299382913,...,0,0,28350454751,993351,66161,1903297,159674,37.90,55.87,2015


In [5]:
df["covid"] = df["year"].astype(int).isin([2020, 2021]).astype(int)   # explicit anomaly flag

feature_cols = sorted(c for c in df.columns if c not in DROP_COLS)
print(len(feature_cols), "features:")
print(feature_cols)

19 features:
['avg_employment', 'bachelor_or_higher_pct', 'covid', 'establishment_count', 'land_area_sqmi', 'latitude', 'longitude', 'median_age', 'median_household_income', 'n_events', 'n_festivals', 'n_setlistfm', 'n_ticketmaster', 'population', 'quarter', 'total_est_attendance', 'total_expected_attendance', 'total_festival_attendance', 'total_wages_usd']


In [6]:
def split_xy(years):
    d = df.loc[df["year"].astype(int).isin(years)]
    X = d[feature_cols].astype(float)
    y = d[TARGET].astype(float)
    return X, y, (np.log1p(y) if LOG_TARGET else y)

train_data, ytr, ytr_fit = split_xy(TRAIN_YEARS)
validate_data, yva, yva_fit = split_xy(VAL_YEARS)
test_data, yte, _ = split_xy(TEST_YEARS)
print(f"train={len(train_data)}  val={len(validate_data)}  test={len(test_data)}")
assert min(len(train_data), len(validate_data), len(test_data)) > 0, "a split is empty — check the year lists"

train=1624  val=232  test=232


In [7]:
# create local stoage for split up data
model_data_path = Path(Path.cwd().parent, "data", "model", "xgboost")

train_file = model_data_path / "training_data.csv"
train_data.to_csv(train_file, index=False, header=False)

validation_file = model_data_path / "validation_data.csv"
validate_data.to_csv(validation_file, index=False, header=False)

testing_file = model_data_path / "testing_data.csv"
test_data.to_csv(testing_file, index=False, header=False)

# ----- Optional -----
# s3_client = boto3.client("s3")

# Load the data into the s3 bucket as well.  Example call below:
# s3_client.upload_file(train_file, f"s3://{BUCKET}/{PROJECT}/{prefix}/train", "training_data")

In [8]:
"""helper functions for Simple heuristic and insights"""
def simple_heuristic(df):
	df.insert(0, "dummy_truth", (
		((df["median_household_income"] * df["population"] * 0.005) # food baseline
		+
		(df["total_est_attendance"] * 36 * 0.23) # food delta
		).astype(float))
	)
	return

def print_stats(guess, truth):
	df = pd.DataFrame({"dummy": guess["dummy_truth"], "truth": truth})

	df["diff"] = df["dummy"] - df["truth"]
	df["perc"] = df["diff"] / df["truth"]

	print(df["perc"].min(), df["perc"].mean(), df["perc"].max())

In [9]:
DATA_PATH = Path().cwd().parent / "data" / "model" / "xgboost" / "validation_data.csv"
FEATURE_NAMES = ["avg_employment", "bachelor_or_higher_pct", "covid",
       "establishment_count", "land_area_sqmi", "latitude", "longitude",
       "median_age", "median_household_income", "n_events", "n_festivals",
       "n_setlistfm", "n_ticketmaster", "population", "quarter",
       "total_est_attendance", "total_expected_attendance",
       "total_festival_attendance", "total_wages_usd"]

baseline_df = pd.read_csv(DATA_PATH, header=None, names=FEATURE_NAMES)
baseline_df.head()

,avg_employment,bachelor_or_higher_pct,covid,establishment_count,land_area_sqmi,latitude,longitude,median_age,median_household_income,n_events,n_festivals,n_setlistfm,n_ticketmaster,population,quarter,total_est_attendance,total_expected_attendance,total_festival_attendance,total_wages_usd
0,"8,392.00",14.52,0.00,918.00,"1,156.35",39.18,-122.24,35.70,"75,149.00",0.00,0.00,0.00,0.00,"21,895.00",1.00,0.00,0.00,0.00,"101,723,825.00"
1,"47,402.00",13.92,0.00,"4,482.00","1,392.19",36.07,-119.82,32.30,"68,750.00",0.00,0.00,0.00,0.00,"152,830.00",1.00,0.00,0.00,0.00,"581,303,023.00"
2,"53,217.00",17.13,0.00,"4,708.00","2,153.24",37.22,-119.76,34.60,"75,496.00",0.00,0.00,0.00,0.00,"158,790.00",1.00,0.00,0.00,0.00,"640,626,185.00"
3,"16,943.00",22.39,0.00,"1,659.00","1,390.47",36.61,-121.07,35.90,"108,289.00",0.00,0.00,0.00,0.00,"66,056.00",1.00,0.00,0.00,0.00,"225,364,412.00"
4,"78,572.00",31.72,0.00,"8,245.00","1,677.13",39.67,-121.60,36.30,"68,574.00",0.00,0.00,0.00,0.00,"209,470.00",2.00,0.00,0.00,0.00,"1,007,992,805.00"


In [10]:
"""Raw stats of heuristic"""
baseline = baseline_df.copy()
simple_heuristic(baseline)
print_stats(baseline, yva.reset_index(drop=True))

-0.8204624796322187 -0.012166928182805997 2.3336728763589747


In [11]:
"""stats of positive events only"""
with_events = baseline_df.loc[baseline_df["n_events"] > 0]
truth_events = yva.reset_index(drop=True).iloc[with_events.index]

simple_heuristic(with_events)
print_stats(with_events, truth_events)

-0.7633096505681144 -0.10673286798846894 1.2023335661007042


In [12]:
"""stats of negativve events only"""
no_events = baseline_df.loc[baseline_df["n_events"] == 0]
truth_events = yva.reset_index(drop=True).iloc[no_events.index]

simple_heuristic(no_events)
print_stats(no_events, truth_events)

-0.8204624796322187 0.03856049648115891 2.3336728763589747
